In [1]:
import pandas as pd
import os

# Load dataset
import kagglehub
path = kagglehub.dataset_download("stanfordu/stanford-question-answering-dataset")

# List files in path
print("Files in dataset: ", os.listdir(path))



100%|██████████| 8.73M/8.73M [00:00<00:00, 72.4MB/s]

Extracting files...


Files in dataset:  ['dev-v1.1.json', 'train-v1.1.json']


In [2]:
import json

def parse_squad(file_path):
    with open(file_path, 'rb') as f:
        squad_dict = json.load(f)

    data = []
    # Loop through the nested structure
    for group in squad_dict['data']:
        for passage in group['paragraphs']:
            context = passage['context']
            for qa in passage['qas']:
                question = qa['question']
                # Each question can have multiple answers in the dev set
                for answer in qa['answers']:
                    data.append({
                        'context': context,
                        'question': question,
                        'answer_text': answer['text'],
                        'answer_start': answer['answer_start']
                    })
    return pd.DataFrame(data)

# Apply the parser to your files
train_df = parse_squad(os.path.join(path, 'train-v1.1.json'))
dev_df = parse_squad(os.path.join(path, 'dev-v1.1.json'))

print(f"Train rows: {len(train_df)}")
print(train_df.head())

Train rows: 87599
                                             context  \
0  Architecturally, the school has a Catholic cha...   
1  Architecturally, the school has a Catholic cha...   
2  Architecturally, the school has a Catholic cha...   
3  Architecturally, the school has a Catholic cha...   
4  Architecturally, the school has a Catholic cha...   

                                            question  \
0  To whom did the Virgin Mary allegedly appear i...   
1  What is in front of the Notre Dame Main Building?   
2  The Basilica of the Sacred heart at Notre Dame...   
3                  What is the Grotto at Notre Dame?   
4  What sits on top of the Main Building at Notre...   

                               answer_text  answer_start  
0               Saint Bernadette Soubirous           515  
1                a copper statue of Christ           188  
2                        the Main Building           279  
3  a Marian place of prayer and reflection           381  
4       a gol

In [3]:
from transformers import pipeline

# Simple comment: Load a pre-trained model specialized for SQuAD
model_checkpoint = "distilbert-base-cased-distilled-squad"
qa_system = pipeline("question-answering", model=model_checkpoint)

# Testing the "system" with a row from your dataframe
context = train_df.iloc[0]['context']
question = train_df.iloc[0]['question']

# Get the result
result = qa_system(question=question, context=context)

print(f"Question: {question}")
print(f"Answer: {result['answer']}")
print(f"Confidence Score: {result['score']:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Answer: Saint Bernadette Soubirous
Confidence Score: 0.9960


In [4]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

# Define the model names from Hugging Face
# These are specifically fine-tuned on the SQuAD dataset
distilbert_name = "distilbert-base-cased-distilled-squad"
bert_name = "bert-large-uncased-whole-word-masking-finetuned-squad"

# 1. Load Tokenizers
# Tokenizers turn text into IDs that the model understands
distil_tokenizer = AutoTokenizer.from_pretrained(distilbert_name)
bert_tokenizer = AutoTokenizer.from_pretrained(bert_name)

# 2. Load Models
# The 'Auto' class automatically detects the architecture (DistilBERT vs BERT)
distil_model = AutoModelForQuestionAnswering.from_pretrained(distilbert_name)
bert_model = AutoModelForQuestionAnswering.from_pretrained(bert_name)

print("Models and Tokenizers loaded successfully.")

config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-large-uncased-whole-word-masking-finetuned-squad
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models and Tokenizers loaded successfully.


In [5]:
import torch

# Select a sample from your train_df
context = train_df.iloc[0]['context']
question = train_df.iloc[0]['question']

# 1. Tokenize the input (converts text to numbers the model understands)
inputs = distil_tokenizer(question, context, return_tensors="pt")

# 2. Feed the model and get the raw output (logits)
with torch.no_grad():
    outputs = distil_model(**inputs)

# 3. Extract the answer span indices (finding the highest probability)
answer_start_index = torch.argmax(outputs.start_logits)
answer_end_index = torch.argmax(outputs.end_logits)

# 4. Convert the indices back into the actual answer text
predict_answer_tokens = inputs.input_ids[0, answer_start_index : answer_end_index + 1]
predicted_answer = distil_tokenizer.decode(predict_answer_tokens)

print(f"Question: {question}")
print(f"Predicted Answer: {predicted_answer}")

Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Predicted Answer: Saint Bernadette Soubirous


In [6]:
import torch

# Simple comment: Functions to calculate EM and F1
def get_metrics(prediction, truth):
    pred_tokens = prediction.lower().split()
    truth_tokens = truth.lower().split()

    # Exact Match logic
    em = 1 if prediction.lower() == truth.lower() else 0

    # F1 Score logic (overlap)
    common_tokens = set(pred_tokens) & set(truth_tokens)
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        f1 = 1 if pred_tokens == truth_tokens else 0
    elif len(common_tokens) == 0:
        f1 = 0
    else:
        precision = len(common_tokens) / len(pred_tokens)
        recall = len(common_tokens) / len(truth_tokens)
        f1 = 2 * (precision * recall) / (precision + recall)

    return em, f1

# Simple comment: Run evaluation on the first 10 rows of dev_df
results = []
for i in range(10):
    row = dev_df.iloc[i]

    # Extract span manually as we did before
    inputs = distil_tokenizer(row['question'], row['context'], return_tensors="pt")
    with torch.no_grad():
        outputs = distil_model(**inputs)

    start, end = torch.argmax(outputs.start_logits), torch.argmax(outputs.end_logits)
    pred = distil_tokenizer.decode(inputs.input_ids[0, start : end + 1])

    # Compare with ground truth
    em, f1 = get_metrics(pred, row['answer_text'])
    results.append({'EM': em, 'F1': f1})

# Simple comment: Calculate averages
avg_em = sum(r['EM'] for r in results) / len(results)
avg_f1 = sum(r['F1'] for r in results) / len(results)

print(f"Average EM: {avg_em:.2f}")
print(f"Average F1: {avg_f1:.2f}")

Average EM: 0.60
Average F1: 0.72


In [7]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

# Simple comment: Load RoBERTa model and tokenizer
roberta_name = "deepset/roberta-base-squad2"
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_name)
roberta_model = AutoModelForQuestionAnswering.from_pretrained(roberta_name)

print("RoBERTa loaded successfully.")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RoBERTa loaded successfully.


In [8]:
# Simple comment: Evaluation loop for RoBERTa
roberta_results = []

for i in range(10):
    row = dev_df.iloc[i]

    # RoBERTa tokenization
    inputs = roberta_tokenizer(row['question'], row['context'], return_tensors="pt")

    with torch.no_grad():
        outputs = roberta_model(**inputs)

    start = torch.argmax(outputs.start_logits)
    end = torch.argmax(outputs.end_logits)

    pred = roberta_tokenizer.decode(inputs.input_ids[0, start : end + 1])

    em, f1 = get_metrics(pred, row['answer_text'])
    roberta_results.append({'EM': em, 'F1': f1})

avg_roberta_em = sum(r['EM'] for r in roberta_results) / len(roberta_results)
avg_roberta_f1 = sum(r['F1'] for r in roberta_results) / len(roberta_results)

print(f"RoBERTa - Average EM: {avg_roberta_em:.2f}, Average F1: {avg_roberta_f1:.2f}")

RoBERTa - Average EM: 0.00, Average F1: 0.86


In [10]:
!pip install streamlit
import streamlit as st
# Simple comment: Create text boxes for input
passage = st.text_area("Enter Passage")
question = st.text_input("Enter Question")

if st.button("Get Answer"):
    # Simple comment: Call the model we built earlier
    result = qa_system(question=question, context=passage)
    st.write(f"Answer: {result['answer']}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 56.8 MB/s eta 0:00:00
  Attempting uninstall: cachetools
    Found existing installation: cachetools 7.0.1
    Uninstalling cachetools-7.0.1:
      Successfully uninstalled cachetools-7.0.1


2026-02-23 09:07:36.524 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-23 09:07:36.525 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-23 09:07:36.527 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-23 09:07:36.530 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-23 09:07:36.532 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-02-23 09:07:36.533 WARNING streamlit.runtime.scriptrunner_utils.script_run_c

In [11]:
# Simple comment: A loop to let you input questions directly in Colab
def run_cli():
    print("--- QA System (Type 'quit' to stop) ---")
    passage = input("Enter your passage: ")

    while True:
        question = input("\nAsk a question: ")
        if question.lower() == 'quit':
            break

        # Simple comment: Using the DistilBERT logic we built
        inputs = distil_tokenizer(question, passage, return_tensors="pt")
        with torch.no_grad():
            outputs = distil_model(**inputs)

        start = torch.argmax(outputs.start_logits)
        end = torch.argmax(outputs.end_logits)
        answer = distil_tokenizer.decode(inputs.input_ids[0, start : end + 1])

        print(f"Answer: {answer}")

run_cli()

--- QA System (Type 'quit' to stop) ---
Enter your passage: "The Apollo 11 mission landed the first two humans on the Moon on July 20, 1969. Neil Armstrong was the first person to step onto the lunar surface, followed 19 minutes later by Buzz Aldrin. They spent about two and a quarter hours together outside the spacecraft and collected 47.5 pounds of lunar material to bring back to Earth."

Ask a question: How much lunar material did the astronauts collect?
Answer: 47. 5 pounds

Ask a question: When did the apollo mission land 2 humans on the moon?
Answer: July 20, 1969

Ask a question: by much time later did the second person step on the moon?
Answer: 19 minutes

Ask a question: quit
